[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session6/student/Lab4_GraphRAG_KnowledgeGraphs_Student.ipynb)


# Lab 4: GraphRAG — Knowledge Graphs for Clinical Knowledge
## CCI Session 6

**Duration:** 15 minutes

### Clinical Scenario
> Vector RAG is great for 'what is the dose for cisplatin?' but fails for 'what are the relationships between staging, histology, and treatment?' GraphRAG extracts entities and relationships, builds a knowledge graph, and answers multi-hop clinical questions.

### Objective
- Extract clinical entities and relationships from WT guidelines
- Build a knowledge graph with NetworkX
- Visualize the graph
- Query with graph traversal
- Compare with vector RAG on multi-hop questions

---
## Setup

Use Colab **Secrets** for keys. Provide **`WT.pdf`** or reuse **`wt.md`** from Lab 1 to avoid re-parsing.


In [ ]:
!pip install -q networkx matplotlib langchain langchain-openai openai pyvis langchain-community llama-parse faiss-cpu tiktoken

In [ ]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['LLAMA_CLOUD_API_KEY'] = userdata.get('LLAMA_CLOUD_API_KEY')
print('Loaded API keys from Colab secrets.')


## Cell 1 — Define entity and relationship schemas

We use Pydantic to enforce a typed clinical ontology. The LLM will be forced to map free text into these structured types.

In [ ]:
from pydantic import BaseModel
from typing import List, Literal

class Entity(BaseModel):
    name: str
    type: Literal["Disease", "Stage", "Drug", "SideEffect", "Procedure", "Histology", "Outcome"]
    description: str

class Relationship(BaseModel):
    source: str  # entity name
    target: str  # entity name
    type: Literal["TREATS", "CAUSES", "INDICATES", "REQUIRES", "ASSOCIATED_WITH", "PART_OF"]
    description: str

class GraphExtraction(BaseModel):
    entities: List[Entity]
    relationships: List[Relationship]

---
## Cell 2 — Load text chunks from WT.pdf

Use a **small subset** (about 5–10 chunks) for API cost control. Reuse **`/content/wt.md`** from Lab 1 or re-parse **WT.pdf** with LlamaParse (same keys as Lab 1).

**Chunking:** prefer `RecursiveCharacterTextSplitter` around **1500 / 150** so each LLM extraction call sees a coherent guideline section; filter by keywords (`stage`, `chemotherapy`, …) like the solution notebook.


In [ ]:
# TODO: Load 5-10 chunks from the parsed WT.pdf markdown
# Use only the section on staging or treatment for cost control
#
# Hints:
# 1. Read your saved markdown from Lab 1: e.g. open('/content/wt.md').read()
# 2. Filter by keyword (e.g. 'stage', 'treatment', 'chemotherapy') OR slice [start:end]
# 3. Use RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)
# 4. Keep only ~5-10 chunks

from langchain.text_splitter import RecursiveCharacterTextSplitter

# md_text = ...
# splitter = ...
# chunks = ...
# print(f'{len(chunks)} chunks')

## Cell 3 — Extract graph with LLM

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# TODO: For each chunk, use structured output to extract entities + relationships
# llm = ChatOpenAI(model="gpt-4o-mini").with_structured_output(GraphExtraction)
# Aggregate all extractions
#
# Hints:
# 1. Build a system prompt instructing the model to extract clinical entities and relationships
#    that match our schema (Disease, Stage, Drug, ...)
# 2. Loop over chunks, invoke llm with the chunk text, collect .entities and .relationships
# 3. Deduplicate entities by name (lowercased)

all_entities = {}      # name -> Entity
all_relationships = []  # list of Relationship

# llm = ...
# prompt = ...
# for ch in chunks:
#     ex: GraphExtraction = llm.invoke(...)
#     ...

print(f'Entities: {len(all_entities)}, Relationships: {len(all_relationships)}')

## Cell 4 — Build NetworkX graph

In [ ]:
import networkx as nx
G = nx.DiGraph()

# TODO: Add nodes (entities) with attributes (type, description)
# TODO: Add edges (relationships) with type and description
# Print: number of nodes, edges, top-degree nodes
#
# Hints:
# - G.add_node(entity.name, type=entity.type, description=entity.description)
# - G.add_edge(rel.source, rel.target, type=rel.type, description=rel.description)
# - top = sorted(G.degree, key=lambda x: -x[1])[:10]

# print(f'Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}')
# print('Top hubs:', ...)

## Cell 5 — Visualize with pyvis

In [ ]:
from pyvis.network import Network

# TODO: Create interactive graph visualization
# Color nodes by type (Disease=red, Drug=blue, etc.)
#
# Hints:
# color_map = {"Disease":"#e74c3c", "Drug":"#3498db", "Stage":"#9b59b6",
#              "SideEffect":"#f39c12", "Procedure":"#1abc9c",
#              "Histology":"#2ecc71", "Outcome":"#34495e"}
# net = Network(notebook=True, height='600px', width='100%', directed=True, cdn_resources='in_line')
# for n, data in G.nodes(data=True):
#     net.add_node(n, label=n, color=color_map.get(data.get('type'), '#888'),
#                  title=f"{data.get('type')}: {data.get('description','')}")
# for s, t, d in G.edges(data=True):
#     net.add_edge(s, t, label=d.get('type',''))
# net.show('wt_graph.html')
# from IPython.display import IFrame
# IFrame('wt_graph.html', width='100%', height=600)

## Cell 6 — Query the graph (graph traversal)

**Question:** *"What drugs treat Stage III Wilms tumor and what are their side effects?"*

In [ ]:
# TODO: Multi-hop query — "What drugs treat Stage III Wilms tumor and what are their side effects?"
# 1. Find Stage III node (try fuzzy match: any node whose lower-case name contains 'stage iii' or 'stage 3')
# 2. Traverse incoming TREATS edges (drug -TREATS-> stage) to find drugs
# 3. Traverse CAUSES edges from drugs to find side effects (drug -CAUSES-> sideeffect)
# 4. Return structured answer (dict)

def multi_hop_query(G, stage_query='stage iii'):
    # find candidate stage node
    # find drugs treating it
    # find side effects caused by those drugs
    pass

# multi_hop_query(G)

## Cell 7 — Community detection (themes)

In [ ]:
import networkx.algorithms.community as nx_comm

# TODO: Use Louvain or label propagation to find communities
# Each community is a "theme" — print members
#
# Hints:
# UG = G.to_undirected()
# communities = nx_comm.louvain_communities(UG, seed=42)  # or label_propagation_communities
# for i, c in enumerate(communities):
#     print(f'Community {i}: {sorted(c)}')

## Cell 8 — Compare with vector RAG

In [ ]:
# TODO: Same multi-hop question on vector RAG
# Show why vector struggles, why graph wins
#
# Hints:
# - Build a FAISS index from `chunks` with OpenAIEmbeddings
# - Retrieve top 4 for the same question
# - Pass to gpt-4o-mini and ask for a structured answer
# - Observe: vector retrieval may miss the cross-document link between Drug --CAUSES--> SideEffect

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# emb = OpenAIEmbeddings(model='text-embedding-3-small')
# vs = FAISS.from_texts(chunks, emb)
# question = 'What drugs treat Stage III Wilms tumor and what are their side effects?'
# docs = vs.similarity_search(question, k=4)
# ...

## Stretch — Natural-language to graph query

In [ ]:
# Stretch: Add a query function that uses the LLM to translate natural language into graph queries
#
# Idea: Have the LLM return a JSON traversal plan like
# {"start_node": "Stage III", "hops": [{"edge": "TREATS", "direction": "in"}, {"edge": "CAUSES", "direction": "out"}]}
# then execute that plan against G.

## KHCC connection

A KHCC oncology knowledge graph could link tumor boards, regimens, biomarkers, and outcomes — enabling multi-hop questions like *"Which Stage III favorable-histology patients on Regimen DD-4A had relapse and what salvage was used?"* — the kind of reasoning vector RAG cannot do alone.